# Solving `seqdistproject` with the check widget

*Compute evolutionary distances between sequences and cluster them.*

This notebook shows how you solve the project and check your work as you go, using the `%%test` cell magic. Put your function definitions in a `%%test seqdistproject` cell and run it — the checks appear right below, with a ✓ or ✗ per function and any functions you still have to write listed as *not defined yet*.

> Run this notebook from inside its own `seqdistproject/` folder, so the test file `test_seqdistproject.py` and any data files are found.

There is a second way to work: write your functions in the file `seqdistproject.py` and, in a normal cell, run `check("seqdistproject")` — but here we keep everything in the notebook.

In [ ]:
import vscodenb
import im_pytest   # registers the %%test magic and check()

## A first attempt

We start with most of the solution written, but `cluster` still to do. Run the checks and see what's left:

In [ ]:
%%test seqdistproject
# This serve to make the log function available:
from math import log

#############################################################
# Reference solution for the seqdist project.
#############################################################


def sequence_difference(seq1, seq2):
    differences = 0
    for i in range(len(seq1)):
        if seq1[i] != seq2[i]:
            differences += 1
    return differences / len(seq1)


def jukes_cantor(dist):
    return -(3 / 4) * log(1 - (4 / 3) * dist)


def lower_trian_matrix(seq_list):
    lower_trian = []
    for i in range(len(seq_list)):
        row = []
        for j in range(i):
            diff = sequence_difference(seq_list[i], seq_list[j])
            row.append(jukes_cantor(diff))
        lower_trian.append(row)
    return lower_trian


def find_lowest_cell(table):
    x = 1
    y = 0
    min_val = table[x][y]
    for i in range(len(table)):
        for j in range(len(table[i])):
            if table[i][j] < min_val:
                min_val = table[i][j]
                x = i
                y = j
    return [x, y]


def link(x, y):
    # mean, makes this WPGMA (centroid-like linking - not UPGMA)
    return (x + y) / 2


def update_labels(labels, i, j):
    # turn the label at first index into a combination of both labels
    labels[j] = "({},{})".format(labels[j], labels[i])
    # Remove the (now redundant) label in the first index
    del labels[i]


def update_table(table, a, b):
    # For the lower index, reconstruct the entire row (ORANGE)
    for i in range(0, b):
        table[b][i] = link(table[b][i], table[a][i])

    # Link cells to update the column above the min cell (BLUE)
    for i in range(b + 1, a):
        table[i][b] = link(table[i][b], table[a][i])

    # Link cells to update the column below the min cell (RED)
    for i in range(a + 1, len(table)):
        table[i][b] = link(table[i][b], table[i][a])

    # Delete cells we no longer need (lighter colors)
    for i in range(a + 1, len(table)):
        # Remove the (now redundant) first index column entry
        del table[i][a]
    # Remove the (now redundant) first index row
    del table[a]


The widget shows `cluster` as *not defined yet*. Now we add it and run again.

## The finished solution

All functions written — every check should pass:

In [ ]:
%%test seqdistproject
# This serve to make the log function available:
from math import log

#############################################################
# Reference solution for the seqdist project.
#############################################################


def sequence_difference(seq1, seq2):
    differences = 0
    for i in range(len(seq1)):
        if seq1[i] != seq2[i]:
            differences += 1
    return differences / len(seq1)


def jukes_cantor(dist):
    return -(3 / 4) * log(1 - (4 / 3) * dist)


def lower_trian_matrix(seq_list):
    lower_trian = []
    for i in range(len(seq_list)):
        row = []
        for j in range(i):
            diff = sequence_difference(seq_list[i], seq_list[j])
            row.append(jukes_cantor(diff))
        lower_trian.append(row)
    return lower_trian


def find_lowest_cell(table):
    x = 1
    y = 0
    min_val = table[x][y]
    for i in range(len(table)):
        for j in range(len(table[i])):
            if table[i][j] < min_val:
                min_val = table[i][j]
                x = i
                y = j
    return [x, y]


def link(x, y):
    # mean, makes this WPGMA (centroid-like linking - not UPGMA)
    return (x + y) / 2


def update_labels(labels, i, j):
    # turn the label at first index into a combination of both labels
    labels[j] = "({},{})".format(labels[j], labels[i])
    # Remove the (now redundant) label in the first index
    del labels[i]


def update_table(table, a, b):
    # For the lower index, reconstruct the entire row (ORANGE)
    for i in range(0, b):
        table[b][i] = link(table[b][i], table[a][i])

    # Link cells to update the column above the min cell (BLUE)
    for i in range(b + 1, a):
        table[i][b] = link(table[i][b], table[a][i])

    # Link cells to update the column below the min cell (RED)
    for i in range(a + 1, len(table)):
        table[i][b] = link(table[i][b], table[i][a])

    # Delete cells we no longer need (lighter colors)
    for i in range(a + 1, len(table)):
        # Remove the (now redundant) first index column entry
        del table[i][a]
    # Remove the (now redundant) first index row
    del table[a]


def cluster(sequences, names):
    table = lower_trian_matrix(sequences)
    labels = names[:]

    # Until all labels have been joined...
    while len(labels) > 1:
        # Locate lowest cell in the table
        i, j = find_lowest_cell(table)
        # Join the table on the cell co-ordinates
        update_table(table, i, j)
        # Update the labels accordingly
        update_labels(labels, i, j)

    # Return the final label
    return labels[0]